In [ ]:
%matplotlib ipympl

In [ ]:
from __future__ import annotations

import dataclasses
import functools
import typing as tp
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sci_interp
import scipy.optimize as sci_opt
import scipy.signal as sci_sig
from flax import nnx

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_clean_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    data = np.array(pd.read_hdf(file_path))
    return data[:, 1:4], data[:, 4:]

file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-standard-road-v2.hdf"
test_file_path = "/Users/jozbee/work/eng/comp/data/clean_00_sms_drive.hdf"
acc_ref, omega_ref = load_clean_references(file_path)
test_acc_ref, test_omega_ref = load_clean_references(test_file_path)

acc_ref = jnp.clip(acc_ref, -1.0, 1.0)
test_acc_ref = jnp.clip(test_acc_ref, -1.0, 1.0)

## lstq

In [ ]:
@jax.jit
def line_lstq(g: jax.Array) -> jax.Array:
    m = g.size
    ts = jnp.arange(m, dtype=float)
    tmp = m * (m + 1.0)
    A_inv = jnp.array([
        [12 / ((m - 1) * tmp), -6 / tmp],
        [-6 / tmp, 2 * (2 * m - 1) / tmp],
    ])
    B = jnp.array([jnp.dot(ts, g), jnp.sum(g)])
    a, b = A_inv @ B
    return a * ts + b

## convert

In [ ]:
@functools.partial(jax.jit, static_argnames=["m"])
def make_L(g: jax.Array, m: int) -> jax.Array:
    n = g.size
    L = jnp.empty(shape=(n - m + 1, m))

    def body(i: jax.Array, L: jax.Array) -> jax.Array:
        gi = jax.lax.dynamic_slice(g, [i], [m])
        L = L.at[i].set(line_lstq(gi))
        return L

    L = jax.lax.fori_loop(0, L.shape[0], body, L)
    return L

In [ ]:
@functools.partial(jax.jit, static_argnames=["m"])
def make_Lz(g: jax.Array, m: int) -> jax.Array:
    n = g.size
    L = make_L(g, m)

    def Lz(k: int, ell: int) -> jax.Array:
        idx0 = jnp.clip((k - m + 1) + ell, min=0, max=L.shape[0] - 1)
        idx1 = jnp.clip(m - 1 - ell, min=0, max=L.shape[1] - 1)
        return L[idx0, idx1]

    return jnp.fromfunction(Lz, shape=(n, m), dtype=int)

In [ ]:
@functools.partial(jax.jit, static_argnames=["n", "m"])
def make_Hz(n: int, m: int) -> jax.Array:
    def H(k: int, ell: int) -> float:
        zero_cond = (k + ell < m - 1) | (k + ell > n - 1) | (ell < 0) | (ell > m - 1)
        half_cond = (ell == 0) | (ell == m - 1)
        return jax.lax.cond(
            zero_cond,
            lambda: 0.0,
            lambda: jax.lax.cond(
                half_cond,
                lambda: 0.5,
                lambda: 1.0,
            )
        )
    return jnp.fromfunction(H, shape=(n, m), dtype=int)

In [ ]:
@functools.partial(jax.jit, static_argnames=["m"])
def convert_w_y(g: jax.Array, dt: jax.Array, m: int) -> tuple[jax.Array, jax.Array]:
    n = g.size
    Hz = make_Hz(n, m)
    Lz = make_Lz(g, m)
    w = jnp.sum(Hz, axis=1)
    y = jnp.sum(Hz * Lz, axis=1) / w
    return w * dt, y

## viz

In [ ]:
dt = 0.005
data = acc_ref[: 200 * 150, 0]
ts = jnp.arange(data.size) * dt
m = 110
w, y = convert_w_y(data, dt, m)
lam = 0.001
new_ref = sci_interp.make_smoothing_spline(ts, y, w=w, lam=lam)(ts)

old_w = np.ones(data.size) * dt
old_w[0] = old_w[0] * 0.5
old_w[-1] = old_w[-1] * 0.5
old_ref = sci_interp.make_smoothing_spline(ts, data, w=old_w, lam=lam)(ts)

med_ref = sci_sig.medfilt(data, kernel_size=101)

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.plot(ts, data, label="data", alpha=0.2)
ax.plot(ts, old_ref, label="old_ref", alpha=0.5)
ax.plot(ts, med_ref, label="med_ref", alpha=0.5)
ax.plot(ts, new_ref, label="new_ref")
ax.grid()
ax.legend()